# 02b – Context-stream ablation and graph-only baseline (five-fold cold-drug protocol)
**Extends:** Section 5.4 / Table 9 (ablation) of the revised manuscript. **Does not change** notebook `02_ablation_study.ipynb` or any shipped result.

**Why:** In Table 9 the ChemBERTa and cell-line streams were removed *together* (`structure_only_no_cb_cell`), so their separate contributions could not be told apart,
and the "No IMCA" variant kept the context streams, so there was no graph-only, simple-concatenation baseline. This notebook adds three variants, trained under the
**identical** five cold-drug partitions, label standardization, optimizer, schedule and checkpoint selection as notebook 02 (the code of Cells 4-12 is copied from it unchanged):

| Variant (key in the results file) | What changes relative to the full model | Question it answers |
|---|---|---|
| `no_chemberta` | ChemBERTa streams removed; cell-line stream, GATv2 and IMCA kept | How much does ChemBERTa add on top of structure + cell line? |
| `no_cell_line` | Cell-line stream removed; ChemBERTa streams, GATv2 and IMCA kept | How much does the cell line add on top of structure + ChemBERTa? |
| `graph_concat_only` | IMCA, ChemBERTa and cell-line streams removed: weight-shared GATv2, gated-attention pooling, concatenation of the two pooled vectors | Graph-only simple-concatenation baseline (Reviewer 3, Comment 7, item i) |

Together with the four variants of notebook 02 this gives seven rows. Every variant is trained once per fold (15 runs in total), as in the reported ablation.

**How to run:** `Runtime → Run all` on a GPU runtime. The notebook is **resumable**: after every (variant, fold) run it writes one result row and the test predictions to Drive, and
finished runs are skipped when you run it again, so a disconnected session loses at most the run in progress. Models are written to `checkpoints_retrained/ablation_context/`
and never overwrite the shipped ones. Set `FOLDS_TO_RUN` / `VARIANTS_TO_RUN` (Cell 2) to run a subset.

**Reading the output:** the last cells print (i) the extended Table 9 (the shipped rows are read from `results/ablation_results.csv` and `results/kfold_results_summary.csv`),
(ii) the paired fold-wise PCC differences to the full model, and (iii) the within-cell-line PCC of the new variants and of the full model, which separates drug-pair signal from cell-line offsets.
Standard deviations over folds use n−1. Retraining is not bit-for-bit reproducible (single run per fold, non-deterministic GPU operations); see the note in notebook 02.

In [ ]:
# ===== Cell 1: Environment =====
# Colab only. The package versions used for the reported results are pinned in requirements.txt.
!pip install -q rdkit matplotlib pubchempy torch torchvision torchaudio torch-geometric transformers xgboost

# Standard library
import json, os, sys
from pathlib import Path

# Numerics and data
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_rel

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset

# Graph learning
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.aggr import AttentionalAggregation
from torch_geometric.utils import to_dense_batch

# Chemistry
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

# Pretrained chemical language model (ChemBERTa)
from transformers import AutoTokenizer, AutoModel

# Classical ML and metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import average_precision_score, mean_squared_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

RDLogger.DisableLog("rdApp.*")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.8 MB/s eta 0:00:00


In [ ]:
# ===== Cell 2: Configuration =====
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DualGATSynergy") if IN_COLAB else Path(".")   # same project folder as notebooks 01 and 02
DATA_DIR = ROOT / "data"

RUN_HEAVY = True                     # the data cells below (copied from notebook 02) are always needed here
NEW_VARIANTS = ["no_chemberta", "no_cell_line", "graph_concat_only"]
VARIANTS_TO_RUN = list(NEW_VARIANTS)                 # subset allowed, e.g. ["no_cell_line"]
FOLDS_TO_RUN    = [1, 2, 3, 4, 5]                    # subset allowed, e.g. [1, 2]

CKPT_DIR        = ROOT / "checkpoints_retrained" / "ablation_context"   # written here; shipped checkpoints are never touched
RESULTS_SHIPPED = ROOT / "results" / "ablation_results.csv"            # the 20 reported runs of notebook 02
MAIN_RESULTS    = ROOT / "results" / "kfold_results_summary.csv"       # full model, notebook 01
MAIN_PREDS      = ROOT / "predictions" / "cold_drug_test_predictions_shipped_models.csv"   # full model's test predictions, notebook 01
# One small CSV (row) and one predictions file per finished (variant, fold) run: this is what makes re-runs resumable.
ROWS_DIR  = ROOT / "results_context_ablation" / "rows"
PREDS_DIR = ROOT / "results_context_ablation" / "predictions"

SEED = 42
def set_seed(seed=SEED):
    """Called before every (variant, fold) run, so a run does not depend on what was executed before it (as in notebook 02)."""
    torch.manual_seed(seed)   # also seeds CUDA
    np.random.seed(seed)
set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE} | project folder: {ROOT}")

# Flags of DualGATSynergyAblation (Cell 11). Everything not listed keeps the full model's setting.
VARIANTS = {
    "no_chemberta":      dict(weight_share=True, use_cross_attn=True,  bidirectional=True,  use_chemberta=False, use_cell_line=True),
    "no_cell_line":      dict(weight_share=True, use_cross_attn=True,  bidirectional=True,  use_chemberta=True,  use_cell_line=False),
    "graph_concat_only": dict(weight_share=True, use_cross_attn=False, bidirectional=False, use_chemberta=False, use_cell_line=False),
}
LABELS = {"full": "Full DualGATSynergy",
          "no_weight_sharing": "No weight sharing",
          "no_cross_attention_IMCA": "No IMCA (concatenation)",
          "unidirectional_cross_attn": "Unidirectional IMCA",
          "structure_only_no_cb_cell": "Structure-only (no CB, no cell)",
          "no_chemberta": "No ChemBERTa (cell line kept)",
          "no_cell_line": "No cell line (ChemBERTa kept)",
          "graph_concat_only": "Graph-only, concatenation"}
ORDER = list(LABELS)
if DEVICE == "cpu":
    print("⚠ No GPU found: training will be very slow. Runtime → Change runtime type → GPU.")

Mounted at /content/drive
device: cuda | project folder: /content/drive/MyDrive/DualGATSynergy


## Data, molecular representations and cold-drug partition (copied unchanged from notebook 02; the cell numbers are those of notebook 02)

In [ ]:
# ===== Cell 4: Load and filter the raw data =====
if RUN_HEAVY:
    # --- 3.1 DrugComb v1.5 ---
    DRUGCOMB_COLS = ["drug_row", "drug_col", "cell_line_name", "synergy_hsa", "study_name", "tissue_name"]
    drugcomb_raw = pd.read_csv(DATA_DIR / "drugcomb_full_v1.5.csv", usecols=DRUGCOMB_COLS, low_memory=False)
    print(f"Raw records:                 {len(drugcomb_raw):,}   (paper: 1,432,351)")

    # Remove non-oncology studies
    NON_ONCOLOGY_KEYWORDS = ["malaria", "covid", "sars", "plasmodium", "tuberculosis"]
    is_non_oncology = (drugcomb_raw["study_name"].astype(str).str.lower()
                       .str.contains("|".join(NON_ONCOLOGY_KEYWORDS), na=False))
    drugcomb_raw = drugcomb_raw[~is_non_oncology].copy()

    # Drop records without an HSA score or a cell-line name (removes nothing in v1.5)
    drugcomb_raw = drugcomb_raw.dropna(subset=["synergy_hsa", "cell_line_name"])
    n_compounds = len(set(drugcomb_raw["drug_row"]) | set(drugcomb_raw["drug_col"]))
    print(f"After filtering:             {len(drugcomb_raw):,}   (paper: 1,432,146) | compounds: {n_compounds:,} (paper: 4,608)")

    # --- 3.2 Compound -> SMILES cache (resolved through PubChem) ---
    cache = pd.read_csv(DATA_DIR / "smiles_cache.csv")        # columns: name, smiles (empty if not resolved)
    drug_to_smiles = dict(zip(cache["name"], cache["smiles"]))
    known_drugs = {name for name, smi in drug_to_smiles.items() if isinstance(smi, str) and smi}
    print(f"Cache: {len(drug_to_smiles)} names, {len(known_drugs)} resolved   (paper: 452 / 418)")

    # Keep only records where BOTH compounds have a SMILES string
    drugcomb = drugcomb_raw[drugcomb_raw["drug_row"].isin(known_drugs) &
                            drugcomb_raw["drug_col"].isin(known_drugs)].copy()
    n_drugs = len(set(drugcomb["drug_row"]) | set(drugcomb["drug_col"]))
    print(f"DrugComb subset: {len(drugcomb):,} records, {n_drugs} drugs   (paper: 61,111 / 276)")

    # --- 3.3 O'Neil ---
    oneil      = pd.read_csv(DATA_DIR / "oneil_summary_idx.csv")
    drugs_meta = pd.read_csv(DATA_DIR / "drugs.csv")
    cells_meta = pd.read_csv(DATA_DIR / "cell_lines.csv")
    drug_map = dict(zip(drugs_meta["id"], drugs_meta["name"]))
    cell_map = dict(zip(cells_meta["id"], cells_meta["name"]))
    oneil["drug_row"]       = oneil["drug_row_idx"].map(drug_map)
    oneil["drug_col"]       = oneil["drug_col_idx"].map(drug_map)
    oneil["cell_line_name"] = oneil["cell_line_idx"].map(cell_map)
    print(f"O'Neil: {len(oneil):,} records   (paper: 22,737) | unmapped names: "
          f"{int(oneil[['drug_row', 'drug_col', 'cell_line_name']].isna().sum().sum())}")

Raw records:                 1,432,351   (paper: 1,432,351)
After filtering:             1,432,146   (paper: 1,432,146) | compounds: 4,608 (paper: 4,608)
Cache: 452 names, 418 resolved   (paper: 452 / 418)
DrugComb subset: 61,111 records, 276 drugs   (paper: 61,111 / 276)
O'Neil: 22,737 records   (paper: 22,737) | unmapped names: 0


In [ ]:
# ===== Cell 5: O'Neil records with SMILES =====
if RUN_HEAVY:
    oneil["smiles_row"] = oneil["drug_row"].map(drug_to_smiles)
    oneil["smiles_col"] = oneil["drug_col"].map(drug_to_smiles)
    oneil_processed = oneil.dropna(subset=["smiles_row", "smiles_col"])[
        ["drug_row", "drug_col", "smiles_row", "smiles_col", "cell_line_name", "synergy_loewe"]].copy()
    oneil_processed.columns = ["drug_1", "drug_2", "smiles_1", "smiles_2", "cell_line", "synergy_score"]
    print(f"O'Neil: {len(oneil_processed):,} of {len(oneil):,} records have SMILES for both drugs   (paper: 22,737 of 22,737)")

O'Neil: 22,737 of 22,737 records have SMILES for both drugs   (paper: 22,737 of 22,737)


In [ ]:
# ===== Cell 6: ChemBERTa embeddings (frozen pretrained model, [CLS] token) =====
if RUN_HEAVY:
    CHEMBERTA_NAME       = "seyonec/ChemBERTa-zinc-base-v1"
    CHEMBERTA_SHIPPED    = DATA_DIR / "drug_to_chemberta.pt"               # 418 compounds x 768 numbers (about 1.3 MB)
    CHEMBERTA_RECOMPUTED = DATA_DIR / "drug_to_chemberta_recomputed.pt"    # never overwrites the shipped file
    RECOMPUTE_CHEMBERTA  = False   # False: load the shipped file (no download, CPU is fine)
                                   # True : recompute with the pretrained model (inference only, no training)

    if not RECOMPUTE_CHEMBERTA:
        drug_to_chemberta = torch.load(CHEMBERTA_SHIPPED, map_location="cpu")
    else:
        tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_NAME)
        chemberta = AutoModel.from_pretrained(CHEMBERTA_NAME).eval().to(DEVICE)
        print("Model revision:", getattr(chemberta.config, "_commit_hash", "unknown"))

        @torch.no_grad()
        def chemberta_embedding(smiles):
            """[CLS]-token embedding of one SMILES string (truncated to 128 tokens)."""
            inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
            return chemberta(**inputs).last_hidden_state[:, 0, :].squeeze(0).cpu()

        drug_to_chemberta = {name: chemberta_embedding(drug_to_smiles[name]) for name in sorted(known_drugs)}
        torch.save(drug_to_chemberta, CHEMBERTA_RECOMPUTED)

    CHEMBERTA_DIM = next(iter(drug_to_chemberta.values())).shape[0]
    print(f"ChemBERTa embeddings: {len(drug_to_chemberta)} compounds | dim = {CHEMBERTA_DIM}   (paper pipeline: 418 | 768)")
    assert set(drug_to_chemberta) == known_drugs, "embedding file and SMILES cache do not match"

ChemBERTa embeddings: 418 compounds | dim = 768   (paper pipeline: 418 | 768)


In [ ]:
# ===== Cell 7: Molecular graphs =====
if RUN_HEAVY:
    NODE_FEATURES = ["atomic_number", "degree", "explicit_valence", "implicit_valence", "is_aromatic", "in_ring",
                     "formal_charge", "num_radical_electrons", "hybridization", "chiral_tag", "atomic_mass"]
    NODE_DIM, EDGE_DIM = len(NODE_FEATURES), 1      # edge feature: bond order (1.0, 1.5, 2.0 or 3.0)

    def _valences(atom):
        """(explicit, implicit) valence. GetExplicitValence/GetImplicitValence produced the reported results;
        newer RDKit versions replace them with GetValence(...)."""
        try:
            return atom.GetExplicitValence(), atom.GetImplicitValence()
        except AttributeError:
            return atom.GetValence(Chem.ValenceType.EXPLICIT), atom.GetValence(Chem.ValenceType.IMPLICIT)

    def smiles_to_graph(smiles):
        """SMILES string -> PyG graph (node features: NODE_FEATURES; edge feature: bond order).
        Returns None if the SMILES is missing or cannot be parsed."""
        if not isinstance(smiles, str) or not smiles:
            return None
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        features = []
        for atom in mol.GetAtoms():
            explicit_valence, implicit_valence = _valences(atom)
            features.append([
                atom.GetAtomicNum(),
                atom.GetDegree(),
                explicit_valence,
                implicit_valence,
                int(atom.GetIsAromatic()),
                int(atom.IsInRing()),
                atom.GetFormalCharge(),
                atom.GetNumRadicalElectrons(),
                int(atom.GetHybridization()),
                int(atom.GetChiralTag()),       # RDKit chiral tag (none / clockwise / counter-clockwise / other)
                atom.GetMass(),
            ])
        x = torch.tensor(features, dtype=torch.float)

        edge_index, edge_attr = [], []
        for bond in mol.GetBonds():
            s, e = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            order = bond.GetBondTypeAsDouble()
            edge_index += [(s, e), (e, s)]           # undirected: both directions
            edge_attr  += [order, order]

        if edge_index:
            edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
            edge_attr  = torch.tensor(edge_attr,  dtype=torch.float).view(-1, 1)
        else:                                         # single-atom molecule
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr  = torch.empty((0, 1),   dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    # Sanity check on all resolved compounds
    graph_check = {name: smiles_to_graph(drug_to_smiles[name]) for name in known_drugs}
    n_failed   = sum(g is None for g in graph_check.values())
    n_no_bonds = sum(g is not None and g.edge_index.shape[1] == 0 for g in graph_check.values())
    print(f"{len(graph_check)} compounds | failed to parse: {n_failed} | single-atom: {n_no_bonds}   (paper: 0 and 0)")
    assert all(g.x.shape[1] == NODE_DIM for g in graph_check.values() if g is not None)
    del graph_check

418 compounds | failed to parse: 0 | single-atom: 2   (paper: 0 and 0)


In [ ]:
# ===== Cell 8: Combine DrugComb (HSA) and O'Neil (Loewe) =====
if RUN_HEAVY:
    drugcomb_unified = (drugcomb.dropna(subset=["synergy_hsa", "cell_line_name"])
                        [["drug_row", "drug_col", "synergy_hsa", "cell_line_name"]]
                        .rename(columns={"drug_row": "drug_1", "drug_col": "drug_2",
                                         "synergy_hsa": "raw_target", "cell_line_name": "cell_line"}))
    drugcomb_unified["dataset_type"] = "drugcomb"

    oneil_unified = (oneil_processed.dropna(subset=["synergy_score"])
                     [["drug_1", "drug_2", "synergy_score", "cell_line"]]
                     .rename(columns={"synergy_score": "raw_target"}))
    oneil_unified["dataset_type"] = "oneil"

    df_combined = pd.concat([drugcomb_unified, oneil_unified], ignore_index=True)

    # One integer per cell line (order of first appearance) and the list of all compounds
    all_cell_lines   = df_combined["cell_line"].unique()
    cell_line_to_idx = {cl: i for i, cl in enumerate(all_cell_lines)}
    n_cell_lines     = len(all_cell_lines)
    all_drugs        = pd.concat([df_combined["drug_1"], df_combined["drug_2"]]).unique()

    print(f"Records: {len(df_combined):,} (paper: 83,848) | cell-line identifiers: {n_cell_lines} (paper: 251) "
          f"| compounds: {len(all_drugs)} (paper: 309)")

Records: 83,848 (paper: 83,848) | cell-line identifiers: 251 (paper: 251) | compounds: 309 (paper: 309)


In [ ]:
# ===== Cell 9: Five-fold cold-drug partition =====
if RUN_HEAVY:
    COHORTS = ["drugcomb", "oneil"]

    def standardize_labels(train_df, val_df, test_df, fold):
        """Cohort-wise z-scoring of the synergy score into column 'label'.
        Mean and SD of each cohort come from the TRAINING rows only and are then applied to that cohort's
        validation and test rows, so nothing from validation or test enters the statistics."""
        scalers = {}
        for cohort in COHORTS:
            mask = train_df["dataset_type"] == cohort
            if mask.sum() == 0:
                print(f"  [warn] fold={fold}: no '{cohort}' samples in train -> scaler skipped")
                scalers[cohort] = None
                continue
            scalers[cohort] = StandardScaler()
            train_df.loc[mask, "label"] = scalers[cohort].fit_transform(train_df.loc[mask, ["raw_target"]])
        for split_df, split_name in ((val_df, "val"), (test_df, "test")):
            for cohort in COHORTS:
                mask = split_df["dataset_type"] == cohort
                if mask.sum() == 0:
                    continue
                if scalers[cohort] is not None:
                    split_df.loc[mask, "label"] = scalers[cohort].transform(split_df.loc[mask, ["raw_target"]])
                else:   # no training statistics for this cohort: its rows cannot be standardized, so they are dropped
                    print(f"  [warn] fold={fold}: dropping {mask.sum()} '{cohort}' rows from {split_name} (no train scaler)")
                    split_df.drop(split_df[mask].index, inplace=True)
        return scalers


    def build_cold_drug_kfold_partition(df_combined, all_drugs, n_folds=5, val_frac=0.15, seed=42):
        """Drugs (not records) are partitioned. All drugs are shuffled once and cut into n_folds disjoint groups;
        each group is the test set of exactly one fold. Within a fold:
          test  = every record containing at least one test drug
          val   = records with a validation drug and no test drug (46 validation drugs, sampled from the non-test drugs)
          train = all other records (no test drug, no validation drug)
        """
        rng = np.random.RandomState(seed)
        shuffled_drugs = rng.permutation(all_drugs)
        test_groups = np.array_split(shuffled_drugs, n_folds)

        folds = []
        for fold_idx, test_drugs_arr in enumerate(test_groups):
            test_drugs = set(test_drugs_arr)
            remaining_drugs = np.array([d for d in all_drugs if d not in test_drugs])
            fold_rng = np.random.RandomState(seed * 1000 + fold_idx)
            n_val = int(val_frac * len(all_drugs))
            val_drugs = set(fold_rng.choice(remaining_drugs, size=min(n_val, len(remaining_drugs)), replace=False))

            is_test_pair  = df_combined["drug_1"].isin(test_drugs) | df_combined["drug_2"].isin(test_drugs)
            is_val_pair   = (~is_test_pair) & (df_combined["drug_1"].isin(val_drugs) | df_combined["drug_2"].isin(val_drugs))
            is_train_pair = ~is_test_pair & ~is_val_pair
            train_df, val_df, test_df = (df_combined[m].copy() for m in (is_train_pair, is_val_pair, is_test_pair))
            scalers = standardize_labels(train_df, val_df, test_df, fold_idx + 1)

            folds.append({"fold": fold_idx + 1, "train_df": train_df, "val_df": val_df, "test_df": test_df,
                          "test_drugs": test_drugs, "val_drugs": val_drugs,
                          "n_test_drugs": len(test_drugs), "n_val_drugs": len(val_drugs), "scalers": scalers})
        return folds


    def verify_partition(folds, all_drugs):
        """Every drug must be in exactly one test set (no omissions, no duplicates)."""
        all_test_drugs = [d for f in folds for d in f["test_drugs"]]
        counts = pd.Series(all_test_drugs).value_counts()
        n_never, n_duplicated = len(all_drugs) - len(counts), int((counts > 1).sum())
        print(f"Test coverage: {len(counts)}/{len(all_drugs)} drugs | omitted: {n_never} | duplicated: {n_duplicated}")
        assert n_never == 0 and n_duplicated == 0, "not a true partition"
        print("✓ partition verified: every drug is in exactly one test set.")


    def check_or_save_json(path, obj, what):
        """First run: save. Later runs (and reviewers): the rebuilt object must equal the shipped file."""
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            assert json.loads(path.read_text(encoding="utf-8")) == obj, f"{what} differs from the shipped file {path.name}"
            print(f"✓ {what} identical to the shipped file")
        else:
            path.write_text(json.dumps(obj, indent=1, ensure_ascii=False), encoding="utf-8")
            print(f"Saved {what} -> {path}")


    folds = build_cold_drug_kfold_partition(df_combined, all_drugs, n_folds=5, seed=42)
    verify_partition(folds, all_drugs)
    check_or_save_json(ROOT / "splits" / "cold_drug_5fold.json",
                       {str(f["fold"]): {"test_drugs": sorted(map(str, f["test_drugs"])),
                                         "val_drugs":  sorted(map(str, f["val_drugs"]))} for f in folds}, "cold-drug split")
    check_or_save_json(ROOT / "splits" / "cell_line_to_idx.json",
                       {str(k): int(v) for k, v in cell_line_to_idx.items()}, "cell-line index")

    print("\nRecords per partition (compare with Table 2):")
    print(pd.DataFrame([{"fold": f["fold"], "train": len(f["train_df"]), "validation": len(f["val_df"]),
                         "test": len(f["test_df"])} for f in folds]).to_string(index=False))

Test coverage: 309/309 drugs | omitted: 0 | duplicated: 0
✓ partition verified: every drug is in exactly one test set.
✓ cold-drug split identical to the shipped file
✓ cell-line index identical to the shipped file

Records per partition (compare with Table 2):
 fold  train  validation  test
    1  31943       16459 35446
    2  44109       12321 27418
    3  31761       20452 31635
    4  37365       16643 29840
    5  33848       22116 27884


In [ ]:
# ===== Cell 10: Dataset =====
if RUN_HEAVY:
    _mol_graph_cache = {}

    def get_cached_graph(smiles):
        """Build each molecular graph once; return a copy so per-sample attributes never leak into the cache."""
        if smiles not in _mol_graph_cache:
            _mol_graph_cache[smiles] = smiles_to_graph(smiles)
        cached = _mol_graph_cache[smiles]
        return cached.clone() if cached is not None else None


    class UnifiedSynergyDataset(Dataset):
        """Sample = (graph of drug 1, graph of drug 2, standardized synergy label).
        A record is kept only if both drugs have a ChemBERTa embedding and a SMILES that RDKit can parse."""
        def __init__(self, dataframe, drug_smiles_dict, drug_chemberta_dict, cell_line_to_idx):
            self.data        = dataframe.reset_index(drop=True)
            self.drug_smiles = drug_smiles_dict
            self.drug_cb     = drug_chemberta_dict
            self.cell_to_idx = cell_line_to_idx

            valid_drugs = {name for name in drug_chemberta_dict
                           if isinstance(drug_smiles_dict.get(name), str)
                           and Chem.MolFromSmiles(drug_smiles_dict[name]) is not None}
            keep = self.data["drug_1"].isin(valid_drugs) & self.data["drug_2"].isin(valid_drugs)
            self.valid_indices = list(self.data.index[keep])
            print(f"Valid samples: {len(self.valid_indices):,} of {len(self.data):,}")

        def __len__(self):
            return len(self.valid_indices)

        def __getitem__(self, idx):
            row = self.data.iloc[self.valid_indices[idx]]
            g1 = get_cached_graph(self.drug_smiles[row["drug_1"]])
            g2 = get_cached_graph(self.drug_smiles[row["drug_2"]])
            g1.cb = self.drug_cb[row["drug_1"]].unsqueeze(0)
            g2.cb = self.drug_cb[row["drug_2"]].unsqueeze(0)
            cl_idx = self.cell_to_idx[row["cell_line"]]
            g1.cell_idx = torch.tensor([cl_idx], dtype=torch.long)
            g2.cell_idx = torch.tensor([cl_idx], dtype=torch.long)
            target = torch.tensor(row["label"], dtype=torch.float)
            return g1, g2, target

## Configurable model and training code (copied unchanged from notebook 02; the cell numbers are those of notebook 02)

In [ ]:
# ===== Cell 11: DualGATSynergy with switchable components =====
if RUN_HEAVY:
    class DualGATSynergyAblation(nn.Module):
        """DualGATSynergy (Cell 10 of notebook 01) with switchable components. With the default flags it is exactly the
        full model: same modules, same names, same parameter count, so the full model's checkpoints load into it.
          weight_share   : True  = one GATv2 encoder for both drugs; False = two independent encoders
          use_cross_attn : True  = IMCA; False = IMCA removed (late fusion by concatenation)
          bidirectional  : True  = IMCA in both directions; False = Drug 1 -> Drug 2 only
          use_chemberta  : ChemBERTa streams on/off
          use_cell_line  : cell-line stream on/off
        """
        def __init__(self, num_atom_features, chemberta_dim, n_cell_lines, hidden_dim=64, heads=2,
                     weight_share=True, use_cross_attn=True, bidirectional=True,
                     use_chemberta=True, use_cell_line=True):
            super().__init__()
            self.weight_share   = weight_share
            self.use_cross_attn = use_cross_attn
            self.bidirectional  = bidirectional and use_cross_attn
            self.use_chemberta  = use_chemberta
            self.use_cell_line  = use_cell_line

            self.conv1 = GATv2Conv(num_atom_features, hidden_dim, heads=heads, edge_dim=1, dropout=0.2)
            self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=1, dropout=0.2)
            if not weight_share:     # second, independent encoder for Drug 2
                self.conv1_b = GATv2Conv(num_atom_features, hidden_dim, heads=heads, edge_dim=1, dropout=0.2)
                self.conv2_b = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=1, dropout=0.2)

            self.pool = AttentionalAggregation(gate_nn=nn.Sequential(
                nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 1)))

            if use_cross_attn:
                self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, dropout=0.1, batch_first=True)
            if use_chemberta:
                self.chemberta_proj = nn.Sequential(
                    nn.Linear(chemberta_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(0.3))
            if use_cell_line:
                self.cell_embedding = nn.Embedding(n_cell_lines, hidden_dim)

            # The two pooled molecular embeddings are always present; the other streams depend on the flags.
            n_streams = 2
            if use_cross_attn:
                n_streams += 2 if self.bidirectional else 1
            if use_chemberta:
                n_streams += 2
            if use_cell_line:
                n_streams += 1
            self.predictor = nn.Sequential(
                nn.Linear(hidden_dim * n_streams, 256), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(128, 64), nn.ReLU(),
                nn.Linear(64, 1))

        def encode_atoms(self, x, edge_index, edge_attr, drug=1):
            conv1, conv2 = (self.conv1, self.conv2) if (drug == 1 or self.weight_share) else (self.conv1_b, self.conv2_b)
            x = F.elu(conv1(x, edge_index, edge_attr=edge_attr))
            x = F.dropout(x, p=0.2, training=self.training)
            return conv2(x, edge_index, edge_attr=edge_attr)

        def forward(self, d1, d2):
            atoms1 = self.encode_atoms(d1.x, d1.edge_index, d1.edge_attr, drug=1)
            atoms2 = self.encode_atoms(d2.x, d2.edge_index, d2.edge_attr, drug=2)
            emb1, emb2 = self.pool(atoms1, d1.batch), self.pool(atoms2, d2.batch)
            streams, w_1to2, w_2to1 = [emb1, emb2], None, None

            if self.use_cross_attn:
                dense1, mask1 = to_dense_batch(atoms1, d1.batch)
                dense2, mask2 = to_dense_batch(atoms2, d2.batch)
                attn_1to2, w_1to2 = self.cross_attention(dense1, dense2, dense2, key_padding_mask=~mask2)
                streams.append(self.pool(attn_1to2[mask1], d1.batch))
                if self.bidirectional:
                    attn_2to1, w_2to1 = self.cross_attention(dense2, dense1, dense1, key_padding_mask=~mask1)
                    streams.append(self.pool(attn_2to1[mask2], d2.batch))
            if self.use_chemberta:
                streams += [self.chemberta_proj(d1.cb), self.chemberta_proj(d2.cb)]
            if self.use_cell_line:
                streams.append(self.cell_embedding(d1.cell_idx))
            return self.predictor(torch.cat(streams, dim=1)), (w_1to2, w_2to1)


    MODEL_CFG = dict(num_atom_features=NODE_DIM, chemberta_dim=CHEMBERTA_DIM, n_cell_lines=n_cell_lines, hidden_dim=64)

    def count_parameters(**flags):
        m = DualGATSynergyAblation(**MODEL_CFG, **flags)
        return sum(p.numel() for p in m.parameters() if p.requires_grad)

    n_full = count_parameters()
    print(f"{'Full DualGATSynergy':26s} {n_full:>9,} trainable parameters   (paper: 260,482)")
    assert n_full == 260_482, "default flags must give exactly the full model"
    for key, flags in VARIANTS.items():
        print(f"{LABELS[key]:26s} {count_parameters(**flags):>9,} trainable parameters")

Full DualGATSynergy          260,482 trainable parameters   (paper: 260,482)
No ChemBERTa (cell line kept)   178,370 trainable parameters
No cell line (ChemBERTa kept)   228,034 trainable parameters
Graph-only, concatenation     96,514 trainable parameters


In [ ]:
# ===== Cell 12: Training and evaluation functions =====
if RUN_HEAVY:
    def evaluate_model(model, loader):
        """RMSE, PCC and the raw predictions / targets of `model` on `loader`."""
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for batch in loader:
                d1, d2, target = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
                pred, _ = model(d1, d2)
                all_preds.extend(pred.cpu().numpy().flatten())
                all_targets.extend(target.cpu().numpy().flatten())
        rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
        pcc, _ = pearsonr(all_targets, all_preds)
        return float(rmse), float(pcc), all_preds, all_targets


    def train_one_fold(model, train_loader, val_loader, checkpoint_path, num_epochs=200, patience_epochs=40):
        """Adam + Smooth-L1. ONLY the validation set drives the LR scheduler, early stopping and checkpoint selection
        (best validation PCC); the test set is never used here. Returns the best validation PCC."""
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
        scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
        criterion = nn.SmoothL1Loss()
        best_val_pcc, epochs_since_improve = -1.0, 0

        for epoch in range(num_epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                d1, d2, targets = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
                optimizer.zero_grad()
                predictions, _ = model(d1, d2)
                loss = criterion(predictions, targets.view(-1, 1))
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)

            if (epoch + 1) % 2 == 0:                          # validate every 2 epochs
                val_rmse, val_pcc, _, _ = evaluate_model(model, val_loader)
                scheduler.step(val_pcc)
                print(f"  Epoch {epoch+1:03d} | Loss: {avg_loss:.4f} | Val RMSE: {val_rmse:.4f} | Val PCC: {val_pcc:.4f}")
                if val_pcc > best_val_pcc:
                    best_val_pcc, epochs_since_improve = val_pcc, 0
                    torch.save(model.state_dict(), checkpoint_path)
                else:
                    epochs_since_improve += 2
                if epochs_since_improve >= patience_epochs:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break
        return best_val_pcc

## Run the new variants (resumable)

In [ ]:
# ===== Cell 13: Train the new variants; the test set is used exactly once per run =====
if RUN_HEAVY:
    TABLE2_SIZES = {   # fold: (train, validation, test, test drugs) records; must equal Table 2 (same partitions as the full model)
        1: (31943, 16459, 35446, 62), 2: (44109, 12321, 27418, 62), 3: (31761, 20452, 31635, 62),
        4: (37365, 16643, 29840, 62), 5: (33848, 22116, 27884, 61)}
    for f in folds:
        got = (len(f["train_df"]), len(f["val_df"]), len(f["test_df"]), f["n_test_drugs"])
        assert got == TABLE2_SIZES[f["fold"]], f"fold {f['fold']}: {got} != {TABLE2_SIZES[f['fold']]}"
    print("✓ partitions match Table 2 (the same partitions as the full model and the other ablation variants)")

    def make_dataset(df):
        return UnifiedSynergyDataset(df, drug_to_smiles, drug_to_chemberta, cell_line_to_idx)

    ROW_COLS = ["config", "fold", "val_pcc", "test_rmse", "test_pcc", "auc", "aupr"]
    for d in (ROWS_DIR, PREDS_DIR, CKPT_DIR):
        d.mkdir(parents=True, exist_ok=True)
    row_path  = lambda variant, k: ROWS_DIR  / f"{variant}__fold{k}.csv"
    pred_path = lambda variant, k: PREDS_DIR / f"{variant}__fold{k}.csv.gz"
    ckpt_path = lambda variant, k: CKPT_DIR  / f"ablation_{variant}_fold{k}.pth"

    for variant in VARIANTS_TO_RUN:
        for f in folds:
            k = f["fold"]
            if k not in FOLDS_TO_RUN:
                continue
            if row_path(variant, k).exists() and pred_path(variant, k).exists():
                print(f"⏭  {variant} | fold {k}: already done")
                continue
            print(f"\n{'='*60}\n{variant} | fold {k}/5\n{'='*60}")

            set_seed(SEED)
            model = DualGATSynergyAblation(**MODEL_CFG, **VARIANTS[variant]).to(DEVICE)
            test_ds      = make_dataset(f["test_df"])
            test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False)
            train_loader = DataLoader(make_dataset(f["train_df"]), batch_size=128, shuffle=True)
            val_loader   = DataLoader(make_dataset(f["val_df"]),   batch_size=128, shuffle=False)
            ckpt = ckpt_path(variant, k)
            best_val_pcc = train_one_fold(model, train_loader, val_loader, ckpt)     # validation set only

            model.load_state_dict(torch.load(ckpt, map_location=DEVICE))             # best-validation checkpoint
            test_rmse, test_pcc, test_preds, test_targets = evaluate_model(model, test_loader)   # the only use of the test set

            kept = f["test_df"].iloc[test_ds.valid_indices]
            assert np.allclose(kept["label"].values, test_targets, atol=1e-5), "rows and labels are misaligned"
            labels_cls = (kept["raw_target"].values > 10).astype(int)                # positive: raw score > 10 (cohort's native scale)
            auc, aupr = roc_auc_score(labels_cls, test_preds), average_precision_score(labels_cls, test_preds)

            # 1) predictions first (needed for the within-cell-line analysis), 2) the result row LAST: a run counts as finished only
            #    when its row exists, and both files are written to a temporary name and renamed, so there are no half-written files.
            tmp_p = pred_path(variant, k).with_name(pred_path(variant, k).name + ".tmp")
            kept[["drug_1", "drug_2", "cell_line", "dataset_type", "raw_target", "label"]].assign(pred=test_preds, fold=k) \
                .to_csv(tmp_p, index=False, compression="gzip")
            os.replace(tmp_p, pred_path(variant, k))
            row = {"config": variant, "fold": k, "val_pcc": best_val_pcc, "test_rmse": test_rmse,
                   "test_pcc": test_pcc, "auc": auc, "aupr": aupr}
            tmp_r = row_path(variant, k).with_suffix(".tmp")
            pd.DataFrame([row])[ROW_COLS].to_csv(tmp_r, index=False)
            os.replace(tmp_r, row_path(variant, k))
            print(f"  ✓ saved | {variant} fold {k} -> Test PCC: {test_pcc:.4f} | RMSE: {test_rmse:.4f} | AUC: {auc:.4f}")

            del model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

✓ partitions match Table 2 (the same partitions as the full model and the other ablation variants)
⏭  no_chemberta | fold 1: already done
⏭  no_chemberta | fold 2: already done
⏭  no_chemberta | fold 3: already done
⏭  no_chemberta | fold 4: already done
⏭  no_chemberta | fold 5: already done
⏭  no_cell_line | fold 1: already done
⏭  no_cell_line | fold 2: already done
⏭  no_cell_line | fold 3: already done
⏭  no_cell_line | fold 4: already done
⏭  no_cell_line | fold 5: already done
⏭  graph_concat_only | fold 1: already done
⏭  graph_concat_only | fold 2: already done
⏭  graph_concat_only | fold 3: already done
⏭  graph_concat_only | fold 4: already done
⏭  graph_concat_only | fold 5: already done


## Extended Table 9 and paired comparison with the full model

In [ ]:
# ===== Cell 14: Extended Table 9 =====
new_paths = sorted(ROWS_DIR.glob("*__fold*.csv"))
new = pd.concat([pd.read_csv(p) for p in new_paths], ignore_index=True) if new_paths else pd.DataFrame(columns=ROW_COLS)
assert RESULTS_SHIPPED.exists() and MAIN_RESULTS.exists(), "shipped results/ablation_results.csv and results/kfold_results_summary.csv are needed"
shipped = pd.read_csv(RESULTS_SHIPPED)
full = pd.read_csv(MAIN_RESULTS).assign(config="full").sort_values("fold")
allruns = pd.concat([d.reindex(columns=ROW_COLS) for d in (full, shipped, new)], ignore_index=True)
assert not allruns.duplicated(["config", "fold"]).any(), "a (variant, fold) run occurs twice"
allruns.to_csv(ROWS_DIR.parent / "ablation_table6_extended_runs.csv", index=False)

full_by_fold = full.set_index("fold")["test_pcc"]
full_sd = full["test_pcc"].std()
print(f"{'variant':32s} {'folds':>5s} {'PCC':>7s} {'SD':>7s} {'RMSE':>7s} {'AUC':>7s} {'AUPR':>7s}")
for key in ORDER:
    d = allruns[allruns["config"] == key].sort_values("fold")
    if len(d):
        print(f"{LABELS[key]:32s} {len(d):5d} {d['test_pcc'].mean():7.4f} {d['test_pcc'].std():7.4f} "
              f"{d['test_rmse'].mean():7.4f} {d['auc'].mean():7.4f} {d['aupr'].mean():7.4f}")

print("\nPaired difference in test PCC to the full model, per fold (variant - full); folds 1..5:")
for key in ORDER[1:]:
    d = allruns[allruns["config"] == key].set_index("fold")["test_pcc"].sort_index()
    common = sorted(set(d.index) & set(full_by_fold.index))
    if len(common) < 2:
        continue
    diff = d.loc[common] - full_by_fold.loc[common]
    t, p = ttest_rel(d.loc[common], full_by_fold.loc[common]) if len(common) == 5 else (float("nan"), float("nan"))
    print(f"{LABELS[key]:32s} " + " ".join(f"{v:+.3f}" for v in diff.values) +
          f" | mean {diff.mean():+.4f} | variant better in {(diff > 0).sum()}/{len(diff)} folds"
          + (f" | paired t = {t:.2f}, p = {p:.3f} (indicative: folds share training data)" if len(common) == 5 else ""))
print(f"\n(SD over folds of the full model: {full_sd:.4f}. Single run per fold: differences well inside this SD are not evidence of an effect.)")

variant                          folds     PCC      SD    RMSE     AUC    AUPR
Full DualGATSynergy                  5  0.3458  0.0731  0.9447  0.6559  0.3416
No weight sharing                    5  0.3332  0.0708  0.9520  0.6562  0.3350
No IMCA (concatenation)              5  0.3437  0.0863  0.9481  0.6427  0.3357
Unidirectional IMCA                  5  0.3473  0.0709  0.9445  0.6495  0.3336
Structure-only (no CB, no cell)      5  0.2053  0.0350  0.9839  0.5842  0.2765
No ChemBERTa (cell line kept)        5  0.3507  0.0682  0.9412  0.6158  0.3022
No cell line (ChemBERTa kept)        5  0.2508  0.0855  0.9726  0.6559  0.3331
Graph-only, concatenation            5  0.1804  0.0683  0.9877  0.5950  0.2726

Paired difference in test PCC to the full model, per fold (variant - full); folds 1..5:
No weight sharing                +0.019 -0.006 -0.016 -0.021 -0.039 | mean -0.0126 | variant better in 1/5 folds | paired t = -1.30, p = 0.262 (indicative: folds share training data)
No IMCA (concaten

## Within-cell-line PCC (separates drug-pair signal from cell-line offsets)
Pearson correlation computed **inside each cell line** and averaged with weights equal to the number of test rows, over cell lines with at least 30 test rows, pooled over the five folds of one model.
The first cell **checks this definition against the full model's predictions and the manuscript's Table 11** (DrugComb 0.192, O'Neil 0.289); if the numbers differ, use the exact code that produced Table 11 instead.

In [ ]:
# ===== Cell 15: Within-cell-line PCC =====
def within_cell_pcc(df, pred_col="pred", min_rows=30):
    """Weighted mean (weights = number of rows) of the PCC computed within each cell line; cell lines with < min_rows rows are skipped."""
    vals, wts = [], []
    for _, g in df.groupby("cell_line"):
        if len(g) >= min_rows and g["label"].std() > 0 and g[pred_col].std() > 0:
            vals.append(pearsonr(g["label"], g[pred_col])[0]); wts.append(len(g))
    return float(np.average(vals, weights=wts)) if vals else float("nan")

def load_new_preds(variant):
    paths = sorted(PREDS_DIR.glob(f"{variant}__fold*.csv.gz"))
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True) if len(paths) == 5 else None

# (a) definition check on the full model (notebook 01 predictions) against Table 11
if MAIN_PREDS.exists():
    fp = pd.read_csv(MAIN_PREDS)
    for cohort, paper in (("drugcomb", 0.192), ("oneil", 0.289)):
        print(f"full model, {cohort:9s}: within-cell-line PCC {within_cell_pcc(fp[fp['dataset_type'] == cohort]):.3f}   (paper, Table 11: {paper})")
else:
    fp = None
    print(f"{MAIN_PREDS} not found: run notebook 01 (evaluate_saved) first to obtain the full model's predictions.")

# (b) all models with predictions available (full model + the new variants)
rows = []
for key in ["full"] + list(VARIANTS):
    preds = fp if key == "full" else load_new_preds(key)
    if preds is None:
        continue
    rows.append({"variant": LABELS[key], "n": len(preds),
                 "PCC (pooled folds)": pearsonr(preds["label"], preds["pred"])[0],
                 "within-cell PCC, all": within_cell_pcc(preds),
                 "within-cell PCC, DrugComb": within_cell_pcc(preds[preds["dataset_type"] == "drugcomb"]),
                 "within-cell PCC, O'Neil":   within_cell_pcc(preds[preds["dataset_type"] == "oneil"])})
if rows:
    out = pd.DataFrame(rows)
    print("\n" + out.round(3).to_string(index=False))
    out.to_csv(ROWS_DIR.parent / "within_cell_line_pcc.csv", index=False)
print("\nNote: shipped variants of notebook 02 saved no predictions; their within-cell-line PCC needs a re-evaluation of their checkpoints (notebook 02, MODE = 'evaluate_saved').")

full model, drugcomb : within-cell-line PCC 0.192   (paper, Table 8: 0.192)
full model, oneil    : within-cell-line PCC 0.289   (paper, Table 8: 0.289)

                      variant      n  PCC (pooled folds)  within-cell PCC, all  within-cell PCC, DrugComb  within-cell PCC, O'Neil
          Full DualGATSynergy 152223               0.345                 0.218                      0.192                    0.289
No ChemBERTa (cell line kept) 152223               0.358                 0.195                      0.156                    0.267
No cell line (ChemBERTa kept) 152223               0.254                 0.213                      0.188                    0.284
    Graph-only, concatenation 152223               0.195                 0.131                      0.066                    0.265

Note: shipped variants of notebook 02 saved no predictions; their within-cell-line PCC needs a re-evaluation of their checkpoints (notebook 02, MODE = 'evaluate_saved').
